In [2]:
# 09_disease_extension - Cell 0: disease names and identifier columns in the raw CircR2Disease files
from pathlib import Path
import re, yaml
import numpy as np
import pandas as pd

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
with open(PROJECT_ROOT / "configs" / "paths.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
D = PROJECT_ROOT / cfg["outputs"]["processed_dir"] / "dataset_v1"
G = D / "new_graph"; G.mkdir(exist_ok=True)
EXT = G / "disease_ext"; EXT.mkdir(exist_ok=True)

# 1. Disease nodes with their number of circRNA-disease pairs
dis = pd.read_csv(D / "diseases.csv")
cda = pd.read_csv(D / "cda_pairs.csv")
dis["n_pairs"] = dis["disease"].map(cda.groupby("disease").size()).fillna(0).astype(int)

def norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", str(s).lower())).strip()
dis["norm"] = dis["disease"].map(norm)
dup = dis[dis.duplicated("norm", keep=False)].sort_values("norm")

print(f"diseases {len(dis)} | pairs {len(cda)} | pairs per disease: min {dis['n_pairs'].min()}, "
      f"median {int(dis['n_pairs'].median())}, max {dis['n_pairs'].max()} | diseases with <=2 pairs: {int((dis['n_pairs'] <= 2).sum())}")
print(f"names that collapse after normalisation: {dup['norm'].nunique()} groups ({len(dup)} names)")
if len(dup):
    print(dup[["disease_id", "disease", "n_pairs", "norm"]].to_string(index=False))

print("\nAll diseases (id, pairs, name), most frequent first:")
for r in dis.sort_values("n_pairs", ascending=False).itertuples():
    print(f"{r.disease_id:>3} {r.n_pairs:>4}  {r.disease}")

dis[["disease_id", "disease", "norm", "n_pairs"]].to_csv(EXT / "diseases_to_map.csv", index=False)

# 2. Do the raw CircR2Disease files carry any disease identifier?
raw = PROJECT_ROOT / cfg["datasets"]["circr2disease"]["dir"]
files = [p for p in sorted(raw.rglob("*")) if p.is_file()]
print(f"\nraw CircR2Disease files ({len(files)}):")
flag = re.compile(r"malacards|mesh|doid|omim|umls|disease", re.I)
for p in files[:20]:
    cols = None
    try:
        if p.suffix.lower() in (".txt", ".csv", ".tsv"):
            cols = pd.read_csv(p, sep=None, engine="python", nrows=0, encoding_errors="replace").columns.tolist()
        elif p.suffix.lower() in (".xlsx", ".xls"):
            cols = pd.read_excel(p, nrows=0).columns.tolist()
    except Exception as e:
        cols = [f"unreadable: {type(e).__name__}"]
    hits = [c for c in (cols or []) if flag.search(str(c))]
    print(f"  {p.relative_to(raw)} | {p.stat().st_size} bytes | columns {cols} | identifier-like: {hits}")

# 3. Observations block (copy everything below this line into the chat)
print("\n===== OBSERVATIONS =====")
print(f"diseases={len(dis)}; pairs={len(cda)}; <=2 pairs={int((dis['n_pairs'] <= 2).sum())}; "
      f"duplicate-name groups={dup['norm'].nunique()}; raw files={len(files)}")

diseases 207 | pairs 2137 | pairs per disease: min 1, median 4, max 148 | diseases with <=2 pairs: 90
names that collapse after normalisation: 0 groups (0 names)

All diseases (id, pairs, name), most frequent first:
 82  148  Hepatocellular carcinoma
 69  143  Gastric cancer
 40  107  Colorectal cancer
 23  102  Breast cancer
136   83  Non-small cell lung cancer
 75   77  Glioma
 20   63  Bladder cancer
144   62  Osteosarcoma
114   44  Lung adenocarcinoma
 31   42  Cervical cancer
168   41  Prostate cancer
149   35  Papillary thyroid carcinoma
145   33  Ovarian cancer
148   31  Pancreatic ductal adenocarcinoma
 74   29  Glioblastoma
 39   28  Colon cancer
190   28  Systemic lupus erythematosus
140   27  Oral squamous cell carcinoma
141   25  Osteoarthritis
115   24  Lung cancer
  4   23  Acute myeloid leukemia
159   23  Preeclampsia
203   21  Uterine leiomyoma
 70   21  Gastric carcinoma
100   20  Intervertebral disc degeneration
 81   20  Hepatoblastoma
 62   20  Esophageal squamous c

## Cell 1

**Objective:** match the 207 disease names to MeSH descriptors automatically (exact name, then entry terms), report coverage by diseases and by pairs, and list the unmatched names, the matches outside the disease tree, and the diseases that collapse onto one descriptor.

In [3]:
# 09_disease_extension - Cell 1: parse MeSH and match the 207 disease names automatically
import xml.etree.ElementTree as ET

MESH_DIR = PROJECT_ROOT / cfg.get("datasets", {}).get("mesh", {}).get("dir", "data/raw/mesh")
xml_path = sorted(MESH_DIR.glob("desc*.xml"))[-1]
print("MeSH file:", xml_path.name, f"{xml_path.stat().st_size / 1e6:.0f} MB")

# 1. Descriptors, tree numbers and every entry term (normalised) -> descriptor id
rows, term2ui = [], {}
for _, el in ET.iterparse(xml_path, events=("end",)):
    if el.tag != "DescriptorRecord":
        continue
    ui = el.findtext("DescriptorUI")
    name = el.findtext("DescriptorName/String")
    trees = [t.text for t in el.findall("TreeNumberList/TreeNumber")]
    rows.append({"ui": ui, "name": name, "trees": ";".join(trees)})
    for t in {name} | {s.text for s in el.findall("ConceptList/Concept/TermList/Term/String")}:
        term2ui.setdefault(norm(t), set()).add(ui)
    el.clear()
mesh = pd.DataFrame(rows).set_index("ui")
mesh.to_csv(EXT / "mesh_descriptors.csv")
print(f"descriptors {len(mesh)} | indexed terms {len(term2ui)}")

# 2. Automatic matching: exact descriptor name, or an entry term of exactly one descriptor
def in_disease_tree(trees):
    return any(t.startswith("C") or t.startswith("F03") for t in trees.split(";") if t)

out = []
for r in dis.itertuples():
    uis = term2ui.get(r.norm, set())
    if len(uis) == 1:
        ui = next(iter(uis)); nm = mesh.at[ui, "name"]
        mt = "exact" if norm(nm) == r.norm else "entry_term"
    elif len(uis) > 1:
        ui, nm, mt = "", " | ".join(sorted(mesh.at[u, "name"] for u in uis)), "ambiguous"
    else:
        ui, nm, mt = "", "", "none"
    trees = mesh.at[ui, "trees"] if ui else ""
    out.append({"disease_id": r.disease_id, "disease": r.disease, "n_pairs": r.n_pairs, "match_type": mt,
                "mesh_ui": ui, "mesh_name": nm, "trees": trees, "disease_tree": bool(ui) and in_disease_tree(trees)})
auto = pd.DataFrame(out)
auto.to_csv(EXT / "disease_mesh_auto.csv", index=False)

# 3. Coverage report
tot, ok = auto["n_pairs"].sum(), auto["disease_tree"]
print("\nmatch types:", auto["match_type"].value_counts().to_dict())
print(f"matched to a disease-tree descriptor: {int(ok.sum())} of {len(auto)} diseases | "
      f"{100 * auto.loc[ok, 'n_pairs'].sum() / tot:.1f}% of pairs")

print("\nmatched but NOT in the disease tree (processes, cells, phenomena):")
print(auto[auto["match_type"].isin(["exact", "entry_term"]) & ~ok][["disease_id", "disease", "mesh_name", "trees"]]
      .assign(trees=lambda d: d["trees"].str.slice(0, 30)).to_string(index=False))

print("\nambiguous (name belongs to several descriptors):")
print(auto[auto["match_type"] == "ambiguous"][["disease_id", "disease", "mesh_name"]].to_string(index=False))

print("\nunmatched (id, pairs, name):")
for r in auto[auto["match_type"] == "none"].sort_values("n_pairs", ascending=False).itertuples():
    print(f"{r.disease_id:>3} {r.n_pairs:>4}  {r.disease}")

print("\ndiseases that share one MeSH descriptor (disease-tree matches only):")
shared = auto[ok].groupby("mesh_ui").filter(lambda g: len(g) > 1)
for ui, g in shared.groupby("mesh_ui"):
    print(f"{g['mesh_name'].iat[0]} [{ui}] <- " + "; ".join(f"{a} ({b})" for a, b in zip(g["disease"], g["n_pairs"])))

# 4. Observations block (copy everything below this line into the chat)
print("\n===== OBSERVATIONS =====")
print(f"mesh={xml_path.name}; descriptors={len(mesh)}; match types={auto['match_type'].value_counts().to_dict()}; "
      f"disease-tree matches={int(ok.sum())}/{len(auto)} ({100 * auto.loc[ok, 'n_pairs'].sum() / tot:.1f}% of pairs); "
      f"descriptors shared by >1 disease={shared['mesh_ui'].nunique()} (covering {len(shared)} diseases)")

MeSH file: desc2026.xml 313 MB
descriptors 31110 | indexed terms 245494

match types: {'entry_term': 86, 'exact': 69, 'none': 52}
matched to a disease-tree descriptor: 150 of 207 diseases | 87.4% of pairs

matched but NOT in the disease tree (processes, cells, phenomena):
 disease_id             disease           mesh_name                          trees
         12        Angiogenesis        Angiogenesis                    G09.330.020
         28 Cellular senescence Cellular Senescence        G04.043;G07.345.124.130
         93    Immunosenescence    Immunosenescence    G07.345.124.390;G12.460.500
        142        Osteogenesis        Osteogenesis G07.345.500.325.377.625.050.50
        184          Senescence               Aging                    G07.345.124

ambiguous (name belongs to several descriptors):
Empty DataFrame
Columns: [disease_id, disease, mesh_name]
Index: []

unmatched (id, pairs, name):
148   31  Pancreatic ductal adenocarcinoma
 70   21  Gastric carcinoma
203   21  

## Cell 2 (markdown)

**Objective:** add the hand-curated crosswalk, check that every descriptor name exists in the loaded MeSH file, save the final mapping table, and list the 86 entry-term matches for a spot-check.

In [5]:
# 09_disease_extension - Cell 2a: source entries for the still-uncertain diseases
raw = PROJECT_ROOT / cfg["datasets"]["circr2disease"]["dir"]
ent = pd.read_excel(raw / "The_circRNA-disease_entries.xlsx")
ent["norm"] = ent["Disease Name"].map(norm)

targets = ["Inflammatory response", "Inflammation", "Repeated implantation failure",
           "Aortic valve calcification", "Intervertebral disc disease"]
sub = ent[ent["norm"].isin({norm(t) for t in targets})].sort_values(["norm", "PubMed ID"])

for r in sub.to_dict("records"):
    print(f"[{r['Disease Name']}] {r['circRNA Name']} | PMID {r['PubMed ID']} | {r['Species']} | {r['Expression pattern']}")
    print(f"   title: {str(r['Title'])[:200]}")
    print(f"   desc : {str(r['Brief description'])[:300]}")

print("\n===== OBSERVATIONS =====")
print(f"entries per disease: {sub.groupby('Disease Name').size().to_dict()}")

[Aortic valve calcification] hsa_circ_0021086 | PMID 32369647.0 | Human | Upregulated
   title: Melatonin ameliorates aortic valve calcification via the regulation of circular RNA CircRIC3/miR-204-5p/DPP4 signaling in valvular interstitial cells.
   desc : melatonin ameliorates aortic valve calcification via the regulation of CircRIC3/miR-204-5p/DPP4 signaling in hVICs; therefore, melatonin medication might be considered a novel pharmaceutical strategy for CAVD treatment
[Inflammation] hsa_circ_0004587 | PMID 31864925.0 | Human | Upregulated
   title: CircRNA104250 and lncRNAuc001.dgp.1 Promote the PM 2.5-induced Inflammatory Response by Co-Targeting miR-3607-5p in BEAS-2B Cells
   desc : CircRNA104250 and lncRNAuc001.dgp.1 target miR-3607-5p and affect expression of interleukin 1 receptor 1 (IL1R1), which influences the nuclear factor κB (NF-κB) signaling pathway.
[Inflammation] circRNA_0000638 | PMID 32199215.0 | Human | Downregulated
   title: circ_0000638 inhibits neodymium oxide-i

In [6]:
# 09_disease_extension - Cell 2: hand-curated crosswalk + final disease -> MeSH mapping table
name2ui = {norm(n): u for u, n in mesh["name"].items()}

# disease_id -> MeSH descriptor names (histology + site where MeSH has no single heading).
# "=Disease name" copies the mapping of another disease. Written before any model result.
MANUAL = {
    148: ("Carcinoma, Pancreatic Ductal",), 70: ("Stomach Neoplasms",), 203: ("Leiomyoma", "Uterine Neoplasms"),
    139: ("Melanoma", "Mouth Neoplasms"), 146: ("Endometriosis", "Ovarian Diseases"), 88: ("Hypopharyngeal Neoplasms",),
    165: ("Diabetic Retinopathy",), 137: ("Pituitary Neoplasms",), 32: ("Uterine Cervical Neoplasms",),
    187: ("Femur Head Necrosis",), 21: ("Urinary Bladder Neoplasms",), 8: ("Cataract",),
    197: ("Carcinoma, Squamous Cell", "Tongue Neoplasms"), 64: ("Osteoarthritis, Spine",), 79: ("=Liver fibrosis",),
    94: ("Hemangioma",), 135: ("Azoospermia",), 154: ("Leukomalacia, Periventricular",), 61: ("Esophageal Neoplasms",),
    76: ("Hashimoto Disease",), 116: ("Carcinoma, Squamous Cell", "Lung Neoplasms"), 97: ("=Inflammation",),
    163: ("Varicose Veins",), 22: ("Carcinoma, Transitional Cell", "Urinary Bladder Neoplasms"),
    160: ("Primary Ovarian Insufficiency",), 71: ("Gastrointestinal Stromal Tumors",), 162: ("Sjogren's Syndrome",),
    111: ("Laryngeal Neoplasms",), 50: ("Cataract", "Diabetes Complications"), 178: ("Infertility, Female",),
    164: ("Liver Neoplasms",), 51: ("Diabetic Foot",), 42: ("Melanoma", "Conjunctival Neoplasms"),
    30: ("Adenocarcinoma", "Uterine Cervical Neoplasms"), 5: ("Myocardial Infarction",),
    14: ("Aortic Valve Stenosis", "Calcinosis"), 67: ("Gallbladder Neoplasms",), 68: ("Stomach Neoplasms", "Adenocarcinoma"),
    101: ("=Intervertebral disc degeneration",), 98: ("Influenza, Human",), 108: ("Acute Kidney Injury",),
    128: ("Myocardial Infarction", "Heart Injuries"), 181: ("Carcinoma, Adenoid Cystic", "Salivary Gland Neoplasms"),
    174: ("Radiation Injuries", "Liver Diseases"), 167: ("Prostatic Neoplasms", "Adenocarcinoma"),
    188: ("Stomach Neoplasms", "Adenocarcinoma"),
    192: ("Precursor T-Cell Lymphoblastic Leukemia-Lymphoma",), 191: ("Precursor T-Cell Lymphoblastic Leukemia-Lymphoma",),
    193: ("Osteoarthritis", "Temporomandibular Joint Disorders"), 205: ("Vascular Diseases",),
}
# site-specific overrides of automatic merges that lost the site
OVERRIDE = {140: ("Carcinoma, Squamous Cell", "Mouth Neoplasms"), 110: ("Carcinoma, Squamous Cell", "Laryngeal Neoplasms"),
            89: ("Carcinoma, Squamous Cell", "Hypopharyngeal Neoplasms")}
NO_MAPPING = {85: "cell type, not a disease", 134: "cell-level process, no suitable descriptor"}
UNCERTAIN = {178, 128, 97}          # best available descriptor is only approximate
NON_DISEASE = {12, 28, 93, 142, 184}                   # MeSH process / phenomenon descriptors (kept, flagged)

def resolve(names, final):
    uis, bad = [], []
    for n in names:
        if n.startswith("="):
            src = dis.loc[dis["disease"] == n[1:], "disease_id"]
            if len(src) and final.get(int(src.iat[0])):
                uis += final[int(src.iat[0])]["uis"]
            else:
                bad.append(n)
        elif norm(n) in name2ui:
            uis.append(name2ui[norm(n)])
        elif len(term2ui.get(norm(n), ())) == 1:
            uis.append(next(iter(term2ui[norm(n)])))
        else:
            bad.append(n)
    return sorted(set(uis)), bad

final, notfound = {}, []
for r in auto.itertuples():                                   # 1. automatic matches
    if r.match_type in ("exact", "entry_term"):
        final[r.disease_id] = {"uis": [r.mesh_ui], "source": "auto"}
for did, names in {**MANUAL, **OVERRIDE}.items():            # 2. manual and overrides
    uis, bad = resolve(names, final)
    if bad: notfound.append((did, bad))
    if uis:
        final[did] = {"uis": uis, "source": "override" if did in OVERRIDE else "manual"}
missing = set(dis["disease_id"]) - set(final) - set(NO_MAPPING)
assert not missing, f"diseases without a mapping decision: {sorted(missing)}"

rows = []
for r in dis.itertuples():
    f = final.get(r.disease_id)
    uis = f["uis"] if f else []
    flag = ("unmapped" if not f else "non_disease" if r.disease_id in NON_DISEASE
            else "uncertain" if r.disease_id in UNCERTAIN else "")
    rows.append({"disease_id": r.disease_id, "disease": r.disease, "n_pairs": r.n_pairs,
                 "mesh_uis": ";".join(uis), "mesh_names": " + ".join(mesh.at[u, "name"] for u in uis),
                 "source": f["source"] if f else "none", "flag": flag})
fin = pd.DataFrame(rows)
fin.to_csv(EXT / "disease_mesh_map.csv", index=False)

mapped = fin["mesh_uis"] != ""
print(f"descriptor names not found: {notfound if notfound else 'none'}")
print(f"mapped {int(mapped.sum())}/{len(fin)} diseases | {100 * fin.loc[mapped, 'n_pairs'].sum() / fin['n_pairs'].sum():.1f}% of pairs | "
      f"sources {fin['source'].value_counts().to_dict()} | flags {fin['flag'].value_counts().to_dict()}")
print("unmapped:", fin.loc[~mapped, ["disease_id", "disease", "n_pairs"]].values.tolist())

print("\nentry-term matches to spot-check (disease -> MeSH descriptor):")
for r in fin[fin["source"] == "auto"].merge(auto[auto["match_type"] == "entry_term"][["disease_id"]], on="disease_id").itertuples():
    print(f"{r.disease_id:>3} {r.disease}  ->  {r.mesh_names}")

print("\ndiseases that now share an identical descriptor set:")
grp = fin[mapped & (fin["flag"] != "non_disease")].groupby("mesh_uis").filter(lambda g: len(g) > 1)
for _, g in grp.groupby("mesh_uis"):
    print(f"{g['mesh_names'].iat[0]} <- " + "; ".join(f"{a} ({b})" for a, b in zip(g["disease"], g["n_pairs"])))

print("\n===== OBSERVATIONS =====")
print(f"mapped={int(mapped.sum())}/{len(fin)}; pairs covered={100 * fin.loc[mapped, 'n_pairs'].sum() / fin['n_pairs'].sum():.1f}%; "
      f"not found={len(notfound)}; sources={fin['source'].value_counts().to_dict()}; flags={fin['flag'].value_counts().to_dict()}; "
      f"identical-set groups={grp['mesh_uis'].nunique()} covering {len(grp)} diseases")

descriptor names not found: none
mapped 205/207 diseases | 99.9% of pairs | sources {'auto': 152, 'manual': 50, 'override': 3, 'none': 2} | flags {'': 197, 'non_disease': 5, 'uncertain': 3, 'unmapped': 2}
unmapped: [[85, 'Human aortic smooth muscle cell', 1], [134, 'Neuronal cell injury', 1]]

entry-term matches to spot-check (disease -> MeSH descriptor):
  0 Abdominal aortic aneurysm  ->  Aortic Aneurysm, Abdominal
  1 Acne  ->  Acne Vulgaris
  2 Acute ischemic stroke  ->  Ischemic Stroke
  4 Acute myeloid leukemia  ->  Leukemia, Myeloid, Acute
  6 Acute pancreatitis  ->  Pancreatitis
  9 Alzheimer's disease  ->  Alzheimer Disease
 13 Angiosarcoma  ->  Hemangiosarcoma
 20 Bladder cancer  ->  Urinary Bladder Neoplasms
 23 Breast cancer  ->  Breast Neoplasms
 24 Breast carcinoma  ->  Breast Neoplasms
 26 Cancer  ->  Neoplasms
 27 Cardiovascular disease  ->  Cardiovascular Diseases
 29 Cerebral ischemia  ->  Brain Ischemia
 31 Cervical cancer  ->  Uterine Cervical Neoplasms
 34 Chronic h

**Interpretation:** 205 of 207 diseases are mapped (99.9% of pairs). The 207 nodes reduce to about 182 distinct descriptor sets, because 23 groups of 48 diseases share one set (for example Gastric cancer and Gastric carcinoma, or Bladder cancer and Bladder carcinoma). Fourteen mappings are flagged. The entry-term spot check looked correct; three are only approximate: Liver fibrosis → Liver Cirrhosis, Acute pancreatitis → Pancreatitis (broader), and Idiopathic pulmonary arterial hypertension → Familial Primary Pulmonary Hypertension.

**Decision:** freeze disease_mesh_map.csv as a fixed input. Treat the 7 non-disease and unmapped diseases as having no ontology feature. Report the 7 uncertain mappings as a sensitivity analysis.

## Cell 3 (markdown)

**Objective:** build the 207×207 MeSH semantic-similarity matrix (Wang method, attenuation factor 0.5, as in MTGCDA equations 1-3), save the concept-group table that marks synonym diseases, and inspect the schema of the frozen negatives file so the collision count can be written correctly in Cell 4.

In [7]:
# 09_disease_extension - Cell 3: MeSH semantic similarity (Wang method, as in MTGCDA eq. 1-3) + concept groups
mesh = pd.read_csv(EXT / "mesh_descriptors.csv", index_col="ui").fillna("")
fin = pd.read_csv(EXT / "disease_mesh_map.csv").fillna("").sort_values("disease_id").reset_index(drop=True)
assert fin["disease_id"].tolist() == list(range(len(fin))), "disease_id is not 0..N-1"
n = len(fin)

MU = 0.5                                     # semantic attenuation factor (as in MTGCDA)
tree2ui = {t: ui for ui, row in mesh.iterrows() for t in row["trees"].split(";") if t}

def dag_contrib(ui):
    """descriptor -> contribution: itself 1, each level up multiplies by MU (max over its tree numbers)"""
    out = {ui: 1.0}
    for t in mesh.at[ui, "trees"].split(";"):
        parts = t.split(".") if t else []
        for k in range(1, len(parts)):
            anc = tree2ui.get(".".join(parts[:len(parts) - k]))
            if anc is not None:
                out[anc] = max(out.get(anc, 0.0), MU ** k)
    return out

has_mesh = ((fin["mesh_uis"] != "") & (fin["flag"] != "non_disease")).values
contrib = {}
for r in fin.itertuples():
    if not has_mesh[r.Index]:
        continue
    c = {}
    for ui in r.mesh_uis.split(";"):
        for a, v in dag_contrib(ui).items():
            c[a] = max(c.get(a, 0.0), v)
    contrib[r.disease_id] = c
DV = {d: sum(c.values()) for d, c in contrib.items()}

ids = sorted(contrib)
S = np.zeros((n, n), dtype=np.float32)       # rows of diseases without features stay zero
for i in ids:
    for j in ids:
        if j < i:
            continue
        common = contrib[i].keys() & contrib[j].keys()
        S[i, j] = S[j, i] = sum(contrib[i][a] + contrib[j][a] for a in common) / (DV[i] + DV[j])

np.save(EXT / "disease_mesh_sim.npy", S)
fin["has_mesh"] = has_mesh
fin[["disease_id", "disease", "has_mesh", "flag"]].to_csv(EXT / "disease_mesh_index.csv", index=False)

# concept groups: diseases with an identical descriptor set are the same concept
key = fin["mesh_uis"].where(fin["has_mesh"], "none_" + fin["disease_id"].astype(str))
fin["concept_id"] = pd.factorize(key)[0]
fin[["disease_id", "concept_id"]].to_csv(EXT / "disease_concept_groups.csv", index=False)

# 1. Report
m = np.array(ids)
sub = S[np.ix_(m, m)][np.triu_indices(len(m), 1)]
print(f"diseases with ontology features: {len(ids)}/{n} (zero rows: {n - len(ids)})")
print(f"pairwise similarity: mean {sub.mean():.3f} | median {np.median(sub):.3f} | "
      f"90th {np.percentile(sub, 90):.3f} | 99th {np.percentile(sub, 99):.3f} | exactly 1.0: {100 * (sub > 0.999).mean():.2f}%")
print(f"concepts among diseases with features: {fin.loc[fin['has_mesh'], 'concept_id'].nunique()} (from {len(ids)} diseases)")

spot = [69, 70, 40, 23, 82, 9, 141, 201, 52, 180]
print("\nspot check (similarity):")
print(pd.DataFrame(S[np.ix_(spot, spot)], index=[f"{i} {fin.at[i, 'disease'][:24]}" for i in spot],
                   columns=spot).round(2).to_string())

iu = np.triu_indices(n, 1)
vals = S[iu]; keep = (vals > 0) & (vals < 0.999)
ai, bj, vv = iu[0][keep], iu[1][keep], vals[keep]
print("\nmost similar pairs that are not identical:")
for k in np.argsort(-vv)[:8]:
    print(f"  {vv[k]:.2f}  {fin.at[ai[k], 'disease']}  |  {fin.at[bj[k], 'disease']}")

# 2. Schema of the frozen negatives (needed for the collision count in Cell 4)
neg_files = sorted(D.rglob("negatives*.csv"))
print("\nnegatives files:", [str(p.relative_to(D)) for p in neg_files])
for p in neg_files[:3]:
    head = pd.read_csv(p, nrows=3)
    print(f"{p.name}: rows {sum(1 for _ in open(p, encoding='utf-8')) - 1} | columns {head.columns.tolist()}")
    print(head.to_string(index=False))
print("cda_pairs columns:", cda.columns.tolist())

# 3. Observations block (copy everything below this line into the chat)
print("\n===== OBSERVATIONS =====")
print(f"features={len(ids)}/{n}; sim mean={sub.mean():.3f}, median={np.median(sub):.3f}, p99={np.percentile(sub, 99):.3f}, "
      f"identical={100 * (sub > 0.999).mean():.2f}%; concepts={fin.loc[fin['has_mesh'], 'concept_id'].nunique()}; "
      f"negatives files={[p.name for p in neg_files]}")

diseases with ontology features: 200/207 (zero rows: 7)
pairwise similarity: mean 0.039 | median 0.000 | 90th 0.122 | 99th 0.429 | exactly 1.0: 0.14%
concepts among diseases with features: 174 (from 200 diseases)

spot check (similarity):
                               69    70    40    23    82   9     141   201   52    180
69 Gastric cancer             1.00  1.00  0.32  0.17  0.19  0.0  0.00  0.00  0.00  0.00
70 Gastric carcinoma          1.00  1.00  0.32  0.17  0.19  0.0  0.00  0.00  0.00  0.00
40 Colorectal cancer          0.32  0.32  1.00  0.14  0.13  0.0  0.00  0.00  0.00  0.00
23 Breast cancer              0.17  0.17  0.14  1.00  0.16  0.0  0.00  0.00  0.00  0.04
82 Hepatocellular carcinoma   0.19  0.19  0.13  0.16  1.00  0.0  0.00  0.00  0.00  0.00
9 Alzheimer's disease         0.00  0.00  0.00  0.00  0.00  1.0  0.00  0.00  0.00  0.00
141 Osteoarthritis            0.00  0.00  0.00  0.00  0.00  0.0  1.00  0.00  0.00  0.49
201 Type 2 diabetes mellitus  0.00  0.00  0.00  0.00  0.0

**Interpretation:**

- 200 of 207 diseases have ontology features. The 7 zero rows are the 5 process-type and 2 unmapped diseases, as planned.
- The matrix is sparse: the median pairwise similarity is 0, the mean 0.039, the 90th percentile 0.12. More than half of all disease pairs share no MeSH ancestor. I predicted last time that the shared Neoplasms ancestors would lift the mean well above zero. That was wrong: cancers are similar within a branch (Gastric 0.32 with Colorectal) but cross-branch pairs are unrelated.
- The sanity check passes. Gastric cancer matches Gastric carcinoma exactly (1.00), Osteoarthritis sits near Rheumatoid arthritis (0.49), and Type 2 diabetes sits near Diabetic nephropathy (0.21). Alzheimer's is isolated, as expected. The top non-identical pairs are all anatomically sensible (Oral and Tongue SCC 0.85, Breast cancer and Triple negative 0.80).
- "Myocardial infarction | Myocardial injury" at 0.79 is a direct result of the two-descriptor mapping I chose for id 128, which is the intended effect.
- Exactly 1.0 occurs in 0.14% of pairs. That is about 28 pairs, which matches the 24 synonym groups (22 pairs plus two groups of 3). The 200 diseases reduce to 174 concepts.
- negatives_cda.csv has 12,822 rows. That equals 2,137 positives × 2 schemes × 3 seeds, so there is one negative per positive per scheme and seed. This is arithmetic, not something I read from the file, so Cell 4 checks it.

**Decision:** the features are usable. How to feed the sparse 207-dimensional rows into the model (directly into a linear layer, or reduced first) is a choice for later, after Cell 4.

## Cell 4 (markdown)

**Objective:** count how many frozen test negatives are really positives under a synonym name. A strict collision means the circRNA has a known positive for a disease with the same concept. A close collision means a related disease with similarity ≥ 0.75 and below 1. This is a sensitivity analysis only, and the frozen negatives stay untouched.

In [8]:
# 09_disease_extension - Cell 4: frozen test negatives that are positives under a synonym name
from collections import Counter

S = np.load(EXT / "disease_mesh_sim.npy")
grp = pd.read_csv(EXT / "disease_concept_groups.csv")
name2id = dict(zip(dis["disease"], dis["disease_id"]))
id2name = dict(zip(dis["disease_id"], dis["disease"]))
concept = dict(zip(grp["disease_id"], grp["concept_id"]))
neg = pd.read_csv(D / "negatives_cda.csv")

pos = cda.groupby("circbase_id")["disease"].apply(lambda s: [name2id[x] for x in s]).to_dict()
assert neg["disease"].isin(name2id).all() and neg["circbase_id"].isin(pos).all()
assert len(neg) == len(cda) * neg["scheme"].nunique() * neg["seed"].nunique(), "unexpected number of negatives"

strict, maxsim = [], []
for r in neg.itertuples(index=False):
    d, P = name2id[r.disease], pos[r.circbase_id]
    assert d not in P, "a frozen negative is an exact positive"
    strict.append(any(concept[p] == concept[d] for p in P))
    maxsim.append(float(S[d, P].max()))
neg["strict"], neg["max_sim"] = strict, maxsim
neg["close"] = (~neg["strict"]) & (neg["max_sim"] >= 0.75)
neg.to_csv(EXT / "negatives_collision_flags.csv", index=False)      # derived table; the frozen file is untouched

# 1. Per scheme and seed
summ = neg.groupby(["scheme", "seed"]).agg(n=("strict", "size"), strict=("strict", "sum"), close=("close", "sum"))
summ["strict_%"] = (100 * summ["strict"] / summ["n"]).round(2)
summ["close_%"] = (100 * summ["close"] / summ["n"]).round(2)
print(summ.to_string())

# 2. Unique pairs and circRNAs affected
for sch, g in neg.groupby("scheme"):
    u = g.drop_duplicates(["circbase_id", "disease"])
    print(f"\n{sch}: unique negative pairs {len(u)} | strict {int(u['strict'].sum())} | close {int(u['close'].sum())} | "
          f"circRNAs with a strict collision {g.loc[g['strict'], 'circbase_id'].nunique()} of {g['circbase_id'].nunique()}")
    print("  max similarity to a known positive, quantiles 50/90/99:",
          np.percentile(g["max_sim"], [50, 90, 99]).round(2).tolist())

# 3. Which synonym pairs cause the strict collisions (degree_matched)
cnt = Counter()
for r in neg[neg["strict"] & (neg["scheme"] == "degree_matched")].itertuples(index=False):
    d = name2id[r.disease]
    for p in pos[r.circbase_id]:
        if concept[p] == concept[d]:
            cnt[(r.disease, id2name[p])] += 1
print("\ntop causes (negative disease <- circRNA's positive disease), degree_matched, all seeds:")
for (a, b), c in cnt.most_common(12):
    print(f"  {c:>4}  {a}  <-  {b}")

# 4. Observations block (copy everything below this line into the chat)
print("\n===== OBSERVATIONS =====")
for sch, g in neg.groupby("scheme"):
    print(f"{sch}: strict {100 * g['strict'].mean():.2f}% ({int(g['strict'].sum())}/{len(g)}) | "
          f"close {100 * g['close'].mean():.2f}% | strict circRNAs {g.loc[g['strict'], 'circbase_id'].nunique()}")
print(f"rows={len(neg)}; schemes={sorted(neg['scheme'].unique())}; seeds={sorted(neg['seed'].unique())}")

                        n  strict  close  strict_%  close_%
scheme         seed                                        
degree_matched 0     2137      17     10      0.80     0.47
               1     2137      12      5      0.56     0.23
               2     2137      13      9      0.61     0.42
random         0     2137       2      9      0.09     0.42
               1     2137      14      8      0.66     0.37
               2     2137       6      8      0.28     0.37

degree_matched: unique negative pairs 6104 | strict 38 | close 21 | circRNAs with a strict collision 35 of 1582
  max similarity to a known positive, quantiles 50/90/99: [0.04, 0.26, 0.78]

random: unique negative pairs 6314 | strict 20 | close 23 | circRNAs with a strict collision 15 of 1582
  max similarity to a known positive, quantiles 50/90/99: [0.0, 0.23, 0.71]

top causes (negative disease <- circRNA's positive disease), degree_matched, all seeds:
    13  Gastric carcinoma  <-  Gastric cancer
     7  Gastri

**Interpretation:**
- The frozen negatives contain few synonym collisions. In the degree-matched scheme, 0.66% of negatives (42 of 6,411) are the same concept as a known positive of that circRNA, and 35 of 1,582 circRNAs are affected. In the random scheme the rate is 0.34% (22 of 6,411, 15 circRNAs).
- Close collisions (a related disease with similarity ≥ 0.75) are about 0.4% in both schemes.
- Degree-matched is higher than random, as expected, because popular diseases have the synonym partners. The causes are concentrated: Gastric, Breast and Bladder account for 34 of the 43 attributions (one negative matches two positives, hence 43 for 42). The pair Gastric carcinoma ← Gastric cancer alone gives 13.
- The row count checks out: 12,822 = 2,137 × 3 seeds × 2 schemes, so there is one negative per positive.

**Decision:** report the rates and this bound in one sentence, and skip a recomputation. This rests on the assumption that the contaminated negatives are the worst case, so it is an upper bound and not an estimate. The training sampler uses the same degree-based draw, so its contamination is probably similar. I have not measured it, and at this size a concept-aware sampler is not worth adding

In [9]:
# 09_disease_extension - Cell 5: extension design (amendment) + disease feature file
S = np.load(EXT / "disease_mesh_sim.npy")
idx = pd.read_csv(EXT / "disease_mesh_index.csv")
N = len(idx)
assert S.shape == (N, N) and idx["disease_id"].tolist() == list(range(N)) and np.isfinite(S).all()
assert np.allclose(S, S.T)

has = idx["has_mesh"].values.astype(bool)
assert np.allclose(S[~has], 0) and np.allclose(S[:, ~has], 0)     # no ontology feature -> zero row and column
np.save(EXT / "disease_feat_mesh.npy", S.astype(np.float32))      # one input row per disease
np.save(EXT / "disease_feat_mask.npy", has)

# 1. Do the sparse diseases have a well-supported ontology neighbour?
n_pairs = dis.set_index("disease_id")["n_pairs"].reindex(range(N)).values
sparse, rich = has & (n_pairs <= 2), has & (n_pairs >= 10)
best = np.array([S[i, rich & (np.arange(N) != i)].max() for i in np.where(sparse)[0]])
pairs_with_feat = int(n_pairs[has].sum())
print(f"diseases with features {int(has.sum())}/{N} | pairs on those diseases {pairs_with_feat}/{int(n_pairs.sum())} "
      f"({100 * pairs_with_feat / n_pairs.sum():.1f}%)")
print(f"non-zero entries per feature row: median {int(np.median((S[has] > 0).sum(1)))}, "
      f"min {int((S[has] > 0).sum(1).min())}, max {int((S[has] > 0).sum(1).max())}")
print(f"sparse diseases (<=2 pairs) with features: {int(sparse.sum())} | "
      f"their best similarity to a disease with >=10 pairs: quantiles 25/50/75 = {np.percentile(best, [25, 50, 75]).round(2).tolist()} | "
      f"with a neighbour >= 0.3: {int((best >= 0.3).sum())}")

# 2. Amendment file (never overwrite an existing one)
amend = """# Extension A - ontology-derived disease features
# POST HOC: written after the M1-M3 results were known. Reported separately; graph.yaml and M1-M3 are unchanged.
frozen_inputs:
  disease_mesh_map: graph/disease_ext/disease_mesh_map.csv    # 205/207 diseases mapped to MeSH 2026 (fixed before any model)
  similarity: graph/disease_ext/disease_mesh_sim.npy           # Wang method, attenuation 0.5; uses no circRNA labels
  feature_file: graph/disease_ext/disease_feat_mesh.npy
  negatives: negatives_cda.csv                                 # unchanged
excluded_still: [GIP similarity, circRNA functional similarity, any similarity derived from the association matrix]
variants: [M1_mesh, M2_mesh, M3_mesh]
disease_input: >
  Linear(207 -> 128) on the MeSH similarity row for the 200 diseases with features;
  learned Embedding for the 7 diseases without (5 non-disease concepts, 2 unmapped names)
identical_to_main: [graph, relations, edge splits, folds, seeds, negatives, training schedule, model selection, hidden size]
decision_rule:
  keep_features_if: "M3_mesh - M3 on degree-matched AUROC has a 95% bootstrap CI over host groups that excludes 0"
  ablation: "report M3_mesh - M2_mesh and M2_mesh - M1_mesh as for the main models"
  selection: "validation only; no choice is made on test scores"
sensitivity:
  - "learned residual added to the ontology input for all diseases"
  - "drop the 10 flagged diseases (3 uncertain, 5 non-disease, 2 unmapped; 15 pairs)"
  - "exclude strict synonym collisions (negatives_collision_flags.csv)"
  - "stratify by disease training degree (0-2 pairs vs more)"
stop_rule: "If M3_mesh does not beat M3 by the rule above, report the main results unchanged and state that ontology features did not help."
"""
out = PROJECT_ROOT / "configs" / "graph_ext.yaml"
if out.exists():
    print("\ngraph_ext.yaml already exists - left untouched")
else:
    out.write_text(amend, encoding="utf-8"); print("\nwritten:", out.relative_to(PROJECT_ROOT))

# 3. Observations block (copy everything below this line into the chat)
print("\n===== OBSERVATIONS =====")
print(f"features={int(has.sum())}/{N}; pairs covered={100 * pairs_with_feat / n_pairs.sum():.1f}%; "
      f"sparse with features={int(sparse.sum())}; best-neighbour median={np.median(best):.2f}; "
      f"sparse with neighbour>=0.3: {int((best >= 0.3).sum())}; amendment={'written' if not out.exists() or True else ''}")

diseases with features 200/207 | pairs on those diseases 2129/2137 (99.6%)
non-zero entries per feature row: median 50, min 3, max 133
sparse diseases (<=2 pairs) with features: 83 | their best similarity to a disease with >=10 pairs: quantiles 25/50/75 = [0.21, 0.28, 0.57] | with a neighbour >= 0.3: 38

written: configs\graph_ext.yaml

===== OBSERVATIONS =====
features=200/207; pairs covered=99.6%; sparse with features=83; best-neighbour median=0.28; sparse with neighbour>=0.3: 38; amendment=written


**nterpretation:**

- Features exist for 200 of 207 diseases and cover 2,129 of 2,137 pairs (99.6%). Each feature row has a median of 50 non-zero entries (minimum 3), so every mapped disease has some ontology relatives.
- Of the 90 diseases with 2 or fewer pairs, 83 have features. Their best similarity to a well-supported disease (≥10 pairs) has a median of 0.28. For scale, in the Cell 3 spot check Gastric and Colorectal cancer scored 0.32 and Breast cancer versus Hepatocellular carcinoma about 0.16. So 0.3 means "same organ or branch", and 0.2 means "both neoplasms". Thirty-eight of the 83 (46%) have a neighbour at least that close. The other half have only distant relatives.
- Design and decision rules are saved in configs/graph_ext.yaml, marked post hoc.

**Decision:** run a one-fold smoke test, then the full M1/M2/M3 variants.

## Cell 6

**Objective:** find out which host-gene information the project already has, check whether CircR2Disease’s gene symbols cover your 1,582 circRNAs, and pre-register the two diagnostics in graph_ext.yaml before any score is computed.

In [20]:
# 09_disease_extension - Cell 6: host-gene information available + pre-registration of the two diagnostics
# 1. What the project already holds about host genes

g_G = D / "graph"
print("dataset_v1 files:", sorted(p.name for p in D.glob("*"))[:40])
print("graph files     :", sorted(p.name for p in g_G.glob("*") if p.is_file())[:40])
hits = []
for p in sorted(list(D.glob("*.csv")) + list(G.glob("*.csv"))):
    cols = pd.read_csv(p, nrows=0).columns.tolist()
    flag = [c for c in cols if "gene" in c.lower()]
    print(f"  {p.relative_to(D)} | {cols}")
    if flag:
        hits.append((p.name, flag))
print("files with gene-like columns:", hits if hits else "none")

ncbi = PROJECT_ROOT / cfg["datasets"]["ncbi_grch38"]["dir"]
print("\nNCBI GRCh38 folder:", [p.name for p in sorted(ncbi.rglob("*")) if p.is_file()][:15])

# 2. Gene symbols in the raw CircR2Disease entries, matched to the 1,582 model circRNAs
ids = set(cda["circbase_id"])
ent = pd.read_excel(cfg_raw := PROJECT_ROOT / cfg["datasets"]["circr2disease"]["dir"] / "The_circRNA-disease_entries.xlsx")
syn = ent["Synonyms"].fillna("").astype(str).str.split(r"[;,|\s]+")
keys = [set([str(a)] + [s for s in b if s]) for a, b in zip(ent["circRNA Name"], syn)]
ent["match"] = [next(iter(k & ids), None) for k in keys]
m = ent.dropna(subset=["match"])
with_gene = m[m["Gene Symbol"].notna() & (m["Gene Symbol"].astype(str).str.strip() != "")]
print(f"\nentries {len(ent)} | entries matched to a model circRNA {len(m)} | "
      f"model circRNAs matched {m['match'].nunique()}/{len(ids)} | "
      f"with a gene symbol {with_gene['match'].nunique()}/{len(ids)}")
print("example symbols:", with_gene["Gene Symbol"].astype(str).head(8).tolist())

# 3. Pre-registration (written before any diagnostic score exists; never overwritten)
EXT_YAML = PROJECT_ROOT / "configs" / "graph_ext.yaml"
cur = yaml.safe_load(EXT_YAML.read_text(encoding="utf-8"))
if "diagnostics" not in cur:
    with open(EXT_YAML, "a", encoding="utf-8") as f:
        f.write('''
diagnostics:     # pre-registered; written before any diagnostic score was computed. No parameters are fitted.
  host_gene_prior:
    source: "CTD curated gene-disease associations (DirectEvidence marker/mechanism or therapeutic only; MeSH-coded diseases)"
    score: "for a pair (circRNA c, disease d): the largest MeSH similarity (Wang, attenuation 0.5) between d and any disease directly associated with c's host gene; 0 if the gene has none"
  mirna_enrichment:
    source: "HMDD v3.0 miRNA-disease associations, mapped to the seed families"
    score: "Circ2Traits-style enrichment: -log10 hypergeometric p of the disease's miRNA families among the families bound by c (universe = the graph's miRNA families)"
  evaluation: "AUROC on the frozen degree-matched negatives (negatives_cda.csv), unique pairs; bootstrap over host groups"
  pass_rule: "AUROC >= 0.55 and the lower 95% bound > 0.527 (the length-only floor)"
  if_pass: "add that relation as static edges (outside the 70/30 split, never decoded) and run the pre-registered ablation ladder against M3"
  if_fail: "do not add the relation; report the diagnostic AUROC and its interval"
''')
    print("\ndiagnostics pre-registered in", EXT_YAML.relative_to(PROJECT_ROOT))
else:
    print("\ndiagnostics already pre-registered - left untouched")

print("\n===== OBSERVATIONS =====")
print(f"gene-like columns in project files: {hits if hits else 'none'}; "
      f"model circRNAs with a CircR2Disease gene symbol: {with_gene['match'].nunique()}/{len(ids)}")

dataset_v1 files: ['RNA-FM_embedding_output', 'cda_pairs.csv', 'circrnas.csv', 'diseases.csv', 'folds.csv', 'graph', 'manifest.json', 'mirna_edges.csv', 'mirna_targets.csv', 'negatives_cda.csv', 'new_graph', 'probe_results', 'rbp_edges.csv', 'rbp_targets.csv', 'rules.yaml', 'sequences.csv', 'train_results']
graph files     : ['edges_circrna_disease.csv', 'edges_circrna_mirna.csv', 'edges_circrna_rbp.csv', 'graph_tables.json', 'mirna_families.csv', 'nodes_circrna.csv', 'nodes_disease.csv', 'nodes_mirna_family.csv', 'nodes_rbp.csv', 'x_circrna.npy']
  cda_pairs.csv | ['circbase_id', 'disease']
  circrnas.csv | ['circbase_id', 'host_group', 'seq_len', 'n_diseases', 'mirna_labelled', 'rbp_labelled', 'mirna_has_pos', 'rbp_has_pos', 'flagged_id', 'in_all_three', 'is_hub']
  diseases.csv | ['disease_id', 'disease']
  folds.csv | ['circbase_id', 'fold_seed0', 'fold_seed1', 'fold_seed2']
  mirna_edges.csv | ['circbase_id', 'mirna_id', 'mirna_name', 'clip_max', 'pos_working', 'pos_clip1', 'pos_c

**Interpretation:**

- None of the project’s own tables has a host-gene column. The only gene information is the “Gene Symbol” field in the raw CircR2Disease entries, and the project folder holds only the GRCh38 FASTA file. That has no annotation, so a gene lookup from coordinates is not possible.
- Gene symbols cover 1,492 of 1,582 model circRNAs (94.3%). In the raw table, 76 circRNAs match no entry at all and 14 match an entry without a symbol. For the diagnostic, these 90 get a score of 0.
- The diagnostics are pre-registered in configs/graph_ext.yaml, before any score exists.

**Decision:** proceed with the host-gene diagnostic. Hold HMDD until the first diagnostic is done.

## Cell 7

**Objective:** build the host-gene table, compare gene symbols with host_group, parse the CTD curated gene-disease file, and report coverage. No diagnostic score is computed yet.

In [21]:
# 09_disease_extension - Cell 7: host-gene table, host_group consistency check, CTD curated associations (coverage only)
# 1. Host genes of the model circRNAs from the CircR2Disease entries (matched as in Cell 6)
raw_dir = PROJECT_ROOT / cfg["datasets"]["circr2disease"]["dir"]
ent = pd.read_excel(raw_dir / "The_circRNA-disease_entries.xlsx")
ids = set(cda["circbase_id"])
syn = ent["Synonyms"].fillna("").astype(str).str.split(r"[;,|\s]+")
keys = [set([str(a)] + [s for s in b if s]) for a, b in zip(ent["circRNA Name"], syn)]
ent["circbase_id"] = [next(iter(k & ids), None) for k in keys]
g = ent.dropna(subset=["circbase_id"]).copy()
g["gene"] = g["Gene Symbol"].fillna("").astype(str).str.split(r"[;,/|]")
g = g.explode("gene")
g["gene"] = g["gene"].str.strip().str.upper()
g = g[(g["gene"] != "") & (g["gene"] != "NAN")][["circbase_id", "gene"]].drop_duplicates()

circ = pd.read_csv(D / "circrnas.csv")
hg = circ.set_index("circbase_id")["host_group"]
g["host_group"] = g["circbase_id"].map(hg)
g.to_csv(EXT / "circ_host_genes.csv", index=False)

# 2. Does the gene symbol agree with host_group (the unit the folds are grouped by)?
per_gene = g.groupby("gene")["host_group"].nunique()
per_group = g.groupby("host_group")["gene"].nunique()
single = g.groupby("circbase_id")["gene"].transform("nunique") == 1
eq = (g.loc[single, "host_group"].astype(str).str.upper() == g.loc[single, "gene"]).mean()
print(f"circRNAs with a gene symbol {g['circbase_id'].nunique()} | distinct genes {g['gene'].nunique()} | "
      f"circRNAs with >1 symbol {int((g.groupby('circbase_id')['gene'].nunique() > 1).sum())}")
print(f"host_group equals the gene symbol for {100 * eq:.1f}% of single-symbol circRNAs")
print(f"genes whose circRNAs fall into >1 host group: {int((per_gene > 1).sum())}/{len(per_gene)}")
print(f"host groups containing >1 gene symbol: {int((per_group > 1).sum())}/{len(per_group)}")
bad = per_gene[per_gene > 1].index[:8]
print("examples (gene -> host groups):", {b: sorted(g.loc[g['gene'] == b, 'host_group'].astype(str).unique())[:4] for b in bad})

# 3. CTD curated gene-disease associations
ctd_path = PROJECT_ROOT / "data" / "raw" / "ctd" / "CTD_genes_diseases.tsv.gz"
assert ctd_path.exists(), "download CTD_genes_diseases.tsv.gz into data/raw/ctd first (PowerShell command in the previous message)"
cols = ["GeneSymbol", "GeneID", "DiseaseName", "DiseaseID", "DirectEvidence", "InferenceChemicalName",
        "InferenceScore", "OmimIDs", "PubMedIDs"]
ctd = pd.read_csv(ctd_path, sep="\t", comment="#", names=cols, header=None, dtype=str, keep_default_na=False)
print(f"\nCTD rows {len(ctd)} | DirectEvidence values: {ctd['DirectEvidence'].value_counts().head(6).to_dict()}")

cur = ctd[ctd["DirectEvidence"].str.strip() != ""].copy()                 # curated: marker/mechanism and/or therapeutic
cur["gene"] = cur["GeneSymbol"].str.upper()
cur = cur[cur["DiseaseID"].str.startswith("MESH:")].assign(ui=lambda d: d["DiseaseID"].str[5:])
mesh = pd.read_csv(EXT / "mesh_descriptors.csv", index_col="ui")
cur = cur[cur["ui"].isin(mesh.index)]                                      # descriptors with tree numbers in MeSH 2026
mine = cur[cur["gene"].isin(set(g["gene"]))][["gene", "ui", "DiseaseName", "DirectEvidence", "PubMedIDs"]].drop_duplicates(["gene", "ui"])
mine.to_csv(EXT / "ctd_gene_disease_curated.csv", index=False)

gc = mine.groupby("gene")["ui"].nunique()
circ_cov = g[g["gene"].isin(gc.index)]["circbase_id"].nunique()
print(f"curated MeSH-coded rows {len(cur)} | for our genes {len(mine)} | our genes with >=1 curated disease "
      f"{len(gc)}/{g['gene'].nunique()} | model circRNAs reaching >=1 curated disease {circ_cov}/{len(ids)}")
print(f"diseases per gene: median {int(gc.median())}, 90th {int(gc.quantile(0.9))}, max {int(gc.max())}")

# 4. Observations block (copy everything below this line into the chat)
print("\n===== OBSERVATIONS =====")
print(f"symbols for {g['circbase_id'].nunique()} circRNAs; genes in >1 host_group {int((per_gene > 1).sum())}/{len(per_gene)}; "
      f"host_group==gene {100 * eq:.1f}%; CTD curated reaches {circ_cov}/{len(ids)} circRNAs; "
      f"genes with a curated disease {len(gc)}/{g['gene'].nunique()}")

circRNAs with a gene symbol 1492 | distinct genes 1192 | circRNAs with >1 symbol 8
host_group equals the gene symbol for 98.0% of single-symbol circRNAs
genes whose circRNAs fall into >1 host group: 1/1192
host groups containing >1 gene symbol: 13/1180
examples (gene -> host groups): {'SHPRH': ['EIF4G3', 'SHPRH']}

CTD rows 130077889 | DirectEvidence values: {'': 130043574, 'marker/mechanism': 32044, 'therapeutic': 1961, 'marker/mechanism|therapeutic': 310}
curated MeSH-coded rows 29858 | for our genes 3657 | our genes with >=1 curated disease 627/1192 | model circRNAs reaching >=1 curated disease 810/1582
diseases per gene: median 2, 90th 13, max 129

===== OBSERVATIONS =====
symbols for 1492 circRNAs; genes in >1 host_group 1/1192; host_group==gene 98.0%; CTD curated reaches 810/1582 circRNAs; genes with a curated disease 627/1192


**Interpretation:**

- The fold grouping is clean. Of 1,192 distinct host genes, only 1 (SHPRH) has circRNAs in more than one host group. Its groups are SHPRH and EIF4G3, probably overlapping loci. host_group equals the gene symbol for 98.0% of single-symbol circRNAs, and 13 of 1,180 groups merge several gene symbols, which is the conservative direction. Sibling leakage across folds is negligible, and this strengthens the main benchmark.
- Gene knowledge is available for about half of the circRNAs. CTD has 130 million rows, but nearly all are chemical-inferred. Only 34,315 carry curated evidence (marker/mechanism or therapeutic), and 29,858 of them are MeSH-coded descriptors. For your genes that leaves 3,657 gene-disease pairs. 627 of 1,192 genes (52.6%) have at least one curated disease, and 810 of 1,582 circRNAs (51.2%) reach one.
- The hub-gene worry did not materialise. Diseases per gene have a median of 2, a 90th percentile of 13 and a maximum of 129, so “the best match among the gene’s diseases” can still discriminate. I had expected a long tail of heavily annotated genes.
- The other half of the circRNAs will score 0, because they have no curated gene knowledge. Pairs on those circRNAs will tie. That limits what a pooled AUROC can reach, so I’ll also report the covered circRNAs separately.

**Decision:** run the pre-registered host-gene diagnostic. No model is trained.

## Cell 8

**Objective:** compute the pre-registered host-gene prior for every circRNA-disease pair, evaluate it on the frozen degree-matched negatives (AUROC per seed, averaged, bootstrap over host groups), and apply the pass rule. Controls and sanity checks are listed below.

In [23]:
# 09_disease_extension - Cell 8: host-gene prior diagnostic (pre-registered in configs/graph_ext.yaml; no parameters fitted)
from sklearn.metrics import roc_auc_score

mesh = pd.read_csv(EXT / "mesh_descriptors.csv", index_col="ui").fillna("")
fm = pd.read_csv(EXT / "disease_mesh_map.csv").fillna("").sort_values("disease_id").reset_index(drop=True)
MU = 0.5
tree2ui = {t: ui for ui, row in mesh.iterrows() for t in row["trees"].split(";") if t}

def dag_contrib(ui):
    out = {ui: 1.0}
    for t in mesh.at[ui, "trees"].split(";"):
        parts = t.split(".") if t else []
        for k in range(1, len(parts)):
            anc = tree2ui.get(".".join(parts[:len(parts) - k]))
            if anc is not None:
                out[anc] = max(out.get(anc, 0.0), MU ** k)
    return out

def contrib_of(uis_):
    c = {}
    for ui in uis_:
        for a, v in dag_contrib(ui).items():
            c[a] = max(c.get(a, 0.0), v)
    return c

# 1. Similarity between each of our diseases and each CTD disease (same Wang method as Cell 3)
dis_c = {r.disease_id: contrib_of(r.mesh_uis.split(";")) for r in fm.itertuples() if r.mesh_uis and r.flag != "non_disease"}
dis_dv = {d: sum(c.values()) for d, c in dis_c.items()}
dis_uis = {r.disease_id: set(r.mesh_uis.split(";")) for r in fm.itertuples() if r.mesh_uis and r.flag != "non_disease"}
ctd = pd.read_csv(EXT / "ctd_gene_disease_curated.csv")
hg_ = pd.read_csv(EXT / "circ_host_genes.csv")
uis = sorted(ctd["ui"].unique()); ui_idx = {u: i for i, u in enumerate(uis)}
ui_c = {u: contrib_of([u]) for u in uis}; ui_dv = {u: sum(c.values()) for u, c in ui_c.items()}
n_dis = len(fm); M = np.zeros((n_dis, len(uis)), dtype=np.float32)
for d, cd in dis_c.items():
    for j, u in enumerate(uis):
        common = cd.keys() & ui_c[u].keys()
        if common:
            M[d, j] = sum(cd[a] + ui_c[u][a] for a in common) / (dis_dv[d] + ui_dv[u])

# 2. Disease columns per circRNA (union over its host genes; empty = no curated knowledge -> score 0)
gene_cols = ctd.groupby("gene")["ui"].apply(lambda s: sorted({ui_idx[u] for u in s})).to_dict()
circ = pd.read_csv(D / "circrnas.csv"); all_c = circ["circbase_id"].tolist()
c_cols = {c: np.array([], dtype=int) for c in all_c}
for c, gg in hg_.groupby("circbase_id")["gene"]:
    c_cols[c] = np.array(sorted({j for g_ in gg for j in gene_cols.get(g_, [])}), dtype=int)
name2id = dict(zip(dis["disease"], dis["disease_id"]))
prior = lambda c, d, cols_of=c_cols: float(M[d, cols_of[c]].max()) if len(cols_of[c]) else 0.0

# 3. Pairs: all positives (per seed) + frozen degree-matched negatives (per seed)
neg = pd.read_csv(D / "negatives_cda.csv"); neg = neg[neg["scheme"] == "degree_matched"]
E = pd.concat([cda.assign(y=1, seed=s) for s in range(3)] + [neg[["seed", "circbase_id", "disease"]].assign(y=0)], ignore_index=True)
E["d"] = E["disease"].map(name2id)
Up = E[["circbase_id", "d"]].drop_duplicates()
sc = {(c, d): prior(c, d) for c, d in zip(Up["circbase_id"], Up["d"])}
ex = {(c, d): float(bool(dis_uis.get(d, set()) & {uis[j] for j in c_cols[c]})) for c, d in zip(Up["circbase_id"], Up["d"])}
key = list(zip(E["circbase_id"], E["d"]))
E["prior"] = [sc[k] for k in key]                         # PRE-REGISTERED score
E["exact"] = [ex[k] for k in key]                         # descriptive: the disease itself is among the gene's curated diseases
E["gene_count"] = [len(c_cols[c]) for c in E["circbase_id"]]      # sanity check: circRNA-level only
cov = np.array([len(c_cols[c]) > 0 for c in E["circbase_id"]])
gid, ug = pd.factorize(E["circbase_id"].map(circ.set_index("circbase_id")["host_group"]))

# 4. AUROC per seed, averaged; bootstrap over host groups
def seed_auc(col, w=None, mask=None):
    out = []
    for s in range(3):
        m = E["seed"].values == s
        if mask is not None:
            m = m & mask
        y, x = E["y"].values[m], E[col].values[m]
        ww = None if w is None else w[m]
        if len(np.unique(y)) < 2 or (ww is not None and (ww[y == 1].sum() == 0 or ww[y == 0].sum() == 0)):
            return np.nan
        out.append(roc_auc_score(y, x, sample_weight=ww))
    return float(np.mean(out))

def with_ci(col, mask=None, n=1000, seed=0):
    rng, b = np.random.default_rng(seed), []
    for _ in range(n):
        w = np.bincount(rng.integers(0, len(ug), len(ug)), minlength=len(ug))[gid].astype(float)
        v = seed_auc(col, w, mask)
        if np.isfinite(v):
            b.append(v)
    lo, hi = np.percentile(b, [2.5, 97.5])
    return seed_auc(col, None, mask), lo, hi, len(b)

rows = []
for label, col, mask in (("host-gene prior (PRE-REGISTERED)", "prior", None),
                         ("  same, covered circRNAs only", "prior", cov),
                         ("exact match (descriptive)", "exact", None),
                         ("gene annotation count (sanity: expect ~0.5)", "gene_count", None)):
    a, lo, hi, nb = with_ci(col, mask)
    rows.append({"score": label, "AUROC": round(a, 3), "ci95": f"[{lo:.3f}, {hi:.3f}]", "boot_ok": nb})
print(pd.DataFrame(rows).to_string(index=False))

# 5. Control: give each circRNA another circRNA's gene knowledge (keeps coverage and gene-popularity, destroys the specific link)
rng, ctrl = np.random.default_rng(12345), []
for _ in range(20):
    perm = dict(zip(all_c, rng.permutation(all_c)))
    cols_s = {c: c_cols[perm[c]] for c in all_c}
    dct = {k: prior(k[0], k[1], cols_s) for k in sc}
    E["shuf"] = [dct[k] for k in key]
    ctrl.append(seed_auc("shuf"))
print(f"\nshuffled-gene control: mean AUROC {np.mean(ctrl):.3f} (min {np.min(ctrl):.3f}, max {np.max(ctrl):.3f}) over 20 shuffles")
print(f"pairs whose circRNA has curated gene knowledge: {100 * cov.mean():.1f}% | share with a non-zero prior: {100 * (E['prior'] > 0).mean():.1f}%")

# 6. Pre-registered rule: AUROC >= 0.55 and lower 95% bound > 0.527
a, lo, hi = rows[0]["AUROC"], float(rows[0]["ci95"].strip("[]").split(",")[0]), float(rows[0]["ci95"].strip("[]").split(",")[1])
ok = a >= 0.55 and lo > 0.527
print(f"\nPRE-REGISTERED RULE (AUROC >= 0.55 and lower bound > 0.527): " + ("PASS - the relation may be added" if ok else "NOT met - do not add the relation"))

print("\n===== OBSERVATIONS =====")
for r in rows:
    print(f"{r['score'].strip()}: {r['AUROC']} {r['ci95']}")
print(f"shuffled control {np.mean(ctrl):.3f}; covered pairs {100 * cov.mean():.1f}%; rule passed: {ok}")

                                      score  AUROC           ci95  boot_ok
           host-gene prior (PRE-REGISTERED)  0.507 [0.500, 0.515]     1000
                same, covered circRNAs only  0.512 [0.496, 0.528]     1000
                  exact match (descriptive)  0.505 [0.500, 0.510]     1000
gene annotation count (sanity: expect ~0.5)  0.500 [0.500, 0.500]     1000

shuffled-gene control: mean AUROC 0.502 (min 0.498, max 0.507) over 20 shuffles
pairs whose circRNA has curated gene knowledge: 51.3% | share with a non-zero prior: 31.4%

PRE-REGISTERED RULE (AUROC >= 0.55 and lower bound > 0.527): NOT met - do not add the relation

===== OBSERVATIONS =====
host-gene prior (PRE-REGISTERED): 0.507 [0.500, 0.515]
same, covered circRNAs only: 0.512 [0.496, 0.528]
exact match (descriptive): 0.505 [0.500, 0.510]
gene annotation count (sanity: expect ~0.5): 0.5 [0.500, 0.500]
shuffled control 0.502; covered pairs 51.3%; rule passed: False


**Interpretation:**

- Primary: AUROC 0.507, 95% CI [0.500, 0.515]. The pre-registered rule (AUROC ≥ 0.55 and a lower bound above 0.527) is not met.
- Covered circRNAs only (the half with curated gene knowledge): 0.512 [0.496, 0.528]. The prior is not hidden by the dilution from circRNAs without gene knowledge. The interval is wide and includes 0.5, but its upper end (0.528) is still not clearly above the floor.
- Exact match (the disease itself is among the host gene’s curated diseases): 0.505 [0.500, 0.510].
- Controls behave as they should. The shuffled-gene control (a circRNA given another circRNA’s gene knowledge) scores 0.502 (range 0.498-0.507 over 20 shuffles), so the real prior is not distinguishable from a prior with no link to the circRNA. The annotation-count score is exactly 0.500.
- Coverage: 51.3% of pairs belong to circRNAs with curated gene knowledge, and 31.4% of all pairs have a non-zero prior.

**Decision:** the stop rule applies. The host-gene relation is not added to the graph.

## Cell 9

In [24]:
# 09_disease_extension - Cell 9: HMDD structure and mapping coverage (no score computed)
hm_dir = PROJECT_ROOT / "data" / "raw" / "hmdd"
files = sorted(p for p in hm_dir.glob("*") if p.is_file())
print("HMDD files:", [p.name for p in files]); assert files, "put the HMDD download in data/raw/hmdd first"
p = files[0]
hm = pd.read_excel(p) if p.suffix.lower() in (".xlsx", ".xls") else pd.read_csv(p, sep=None, engine="python", encoding_errors="replace")
print(p.name, hm.shape, hm.columns.tolist()); print(hm.head(3).to_string(index=False))
dcol = next((c for c in hm.columns if "disease" in str(c).lower()), None)
mcol = next((c for c in hm.columns if "mir" in str(c).lower()), None)
print("disease column:", dcol, "| miRNA column:", mcol); assert dcol and mcol, "tell me the column names"

# diseases: HMDD uses MeSH-style names -> match to descriptor names (exact, normalised)
mesh = pd.read_csv(EXT / "mesh_descriptors.csv", index_col="ui").fillna("")
name2ui = {norm(n): u for u, n in mesh["name"].items()}
hm["ui"] = hm[dcol].map(lambda s: name2ui.get(norm(s)))
print(f"\nrows {len(hm)} | distinct diseases {hm[dcol].nunique()} | rows with a MeSH descriptor match {100 * hm['ui'].notna().mean():.1f}%")
fm = pd.read_csv(EXT / "disease_mesh_map.csv").fillna("")
ours = fm[(fm["mesh_uis"] != "") & (fm["flag"] != "non_disease")].copy()
have = ours["mesh_uis"].map(lambda s: bool(set(s.split(";")) & set(hm["ui"].dropna())))
print(f"our diseases with >=1 HMDD record at the exact descriptor level: {int(have.sum())}/{len(ours)} "
      f"({100 * ours.loc[have, 'n_pairs'].sum() / ours['n_pairs'].sum():.1f}% of pairs)")

# miRNAs: HMDD uses precursor names (hsa-mir-155); the graph families keep mature member names
import re
def mkey(s):
    s = re.sub(r"-(5p|3p)$", "", str(s).lower().replace("hsa-", "").strip())
    m = re.match(r"^(mir-\d+[a-z]*|let-7[a-z]*)", s)
    return m.group(1) if m else s
fam = pd.read_csv(g_G / "mirna_families.csv")
print("\nmirna_families.csv columns:", fam.columns.tolist()); print(fam.head(3).to_string(index=False))
members = {mkey(x) for c in fam.columns if any(k in c.lower() for k in ("member", "name", "mirna"))
           for v in fam[c].dropna().astype(str) for x in re.split(r"[;,|]", v) if x.strip()}
hm["key"] = hm[mcol].map(mkey)
print(f"HMDD rows whose miRNA matches a graph-family member: {100 * hm['key'].isin(members).mean():.1f}% | "
      f"distinct matched miRNA keys {hm.loc[hm['key'].isin(members), 'key'].nunique()} of {len(members)} member keys")
per = hm[hm["ui"].notna() & hm["key"].isin(members)].groupby("ui")["key"].nunique()
print(f"matched miRNAs per disease: median {int(per.median())}, 90th {int(per.quantile(0.9))}, max {int(per.max())}")

print("\n===== OBSERVATIONS =====")
print(f"file={p.name}; rows={len(hm)}; disease match={100 * hm['ui'].notna().mean():.1f}%; "
      f"our diseases covered={int(have.sum())}/{len(ours)}; miRNA match={100 * hm['key'].isin(members).mean():.1f}%; "
      f"miRNAs per disease median={int(per.median())}")

HMDD files: ['alldata_v4.txt']
alldata_v4.txt (53553, 5) ['code', 'PMID', 'miRNA', 'disease', 'description']
       code     PMID        miRNA              disease                                                                                                                   description
      other 36948853   hsa-mir-40      Liver Cirrhosis                           MiR-340 mediates the involvement of high mobility group box 1 in the pathogenesis of liver fibrosis
target gene 36948479  hsa-mir-29b Zika Virus Infection Zika virus NS1 suppresses VE-cadherin via hsa-miR-29b-3p/DNMT3b/MMP-9 pathway in human brain microvascular endothelial cells.
    exosome 36948039 hsa-mir-130b        Wound Healing                           ADSCs-derived exosomal H19 accelerates cutaneous wound healing via the miR-130b-3p/PPARγ/STAT3 axis
disease column: disease | miRNA column: miRNA

rows 53553 | distinct diseases 2379 | rows with a MeSH descriptor match 92.7%
our diseases with >=1 HMDD record at the e

**Interpretation:**

- The download worked. HMDD v4.0 (2023.07) has 53,553 records with 2,379 distinct disease names, and 92.7% of the rows match a MeSH descriptor name. Of your 200 diseases with ontology features, 192 (98.6% of pairs) have at least one HMDD record at the exact descriptor level. Disease coverage is good.
- miRNA coverage: 85.9% of HMDD rows name a miRNA that matches a graph-family member. The “475 of 1,194 member keys” figure understates this. My code also collected the MIMAT accession numbers from mirna_families.csv as if they were names, which inflates the denominator. That is my reading of the arithmetic and not something I verified.
- Power: the matched miRNAs per disease have a median of 4, a 90th percentile of 32 and a maximum of 352. Most diseases have only a handful of miRNAs, so an enrichment test has little power for them. The large cancers have long lists, and then almost any circRNA overlaps.
- Two structural limits. HMDD records precursor names such as hsa-mir-29b, while your graph families are built from arm-specific mature miRNAs. A precursor therefore maps to the families of both arms, which differ. The ambiguity is unavoidable and the next cell counts it. The second limit is on the circRNA side: only about half of your circRNAs (835 of 1,582, from the earlier table) have a working miRNA edge, and the rest will score 0.
- One row in the output head looks like a typo in HMDD itself (hsa-mir-40 with a description about miR-340), so some records are noisy.

**Decision:** coverage is sufficient. Run the HMDD diagnostic as the last pre-registered item, with its details fixed first. My expectation is low, for the reason I gave earlier: M3 can already learn miRNA-disease patterns from the graph.

## Cell 10

**Objective:** build the mapping tables (HMDD precursor to seed families, disease to HMDD records, circRNA to bound families), print their structure, and append the pre-registered details to graph_ext.yaml. No enrichment score or AUROC is computed.

In [26]:
# 09_disease_extension - Cell 10: HMDD enrichment diagnostic - mapping tables + pre-registration of the details (no score computed)
import re
from collections import defaultdict

# 1. Graph side: families and the families each circRNA binds (working rule, as in the graph)
me = pd.read_csv(D / "mirna_edges.csv")
me = me[me["pos_working"].astype(bool)].copy()
fam = pd.read_csv(g_G / "mirna_families.csv")
seed_of = dict(zip(fam["mirna_id"], fam["seed"]))
me["seed"] = me["mirna_id"].map(seed_of)
assert me["seed"].notna().all(), "a working edge has no seed family"
universe = sorted(me["seed"].unique())
bound = me.groupby("circbase_id")["seed"].apply(set).to_dict()
print(f"families with a working edge {len(universe)} (graph.yaml: 416) | unique circRNA-family edges {len(me.drop_duplicates(['circbase_id', 'seed']))} (graph.yaml: 7540)")
print(f"circRNAs with >=1 bound family {len(bound)}/{cda['circbase_id'].nunique()} | bound-set size median {int(np.median([len(v) for v in bound.values()]))}, max {max(len(v) for v in bound.values())}")

# 2. HMDD precursor name -> seed families (all arms accepted; ambiguity counted)
def mkey(s):
    s = re.sub(r"-(5p|3p)$", "", str(s).lower().replace("hsa-", "").strip())
    m = re.match(r"^(mir-\d+[a-z]*|let-7[a-z]*)", s)
    return m.group(1) if m else s
uni = set(universe)
key2seeds = defaultdict(set)
for nm, sd in zip(fam["mirna_name"], fam["seed"]):
    if sd in uni:
        key2seeds[mkey(nm)].add(sd)

hm = pd.read_csv(sorted((PROJECT_ROOT / "data" / "raw" / "hmdd").glob("*"))[0], sep=None, engine="python", encoding_errors="replace")
mesh = pd.read_csv(EXT / "mesh_descriptors.csv", index_col="ui").fillna("")
name2ui = {norm(n): u for u, n in mesh["name"].items()}
hm["ui"] = hm["disease"].map(lambda s: name2ui.get(norm(s)))
hm["key"] = hm["miRNA"].map(mkey)
hm["seeds"] = hm["key"].map(lambda k: key2seeds.get(k, set()))
used = hm[hm["ui"].notna() & (hm["seeds"].map(len) > 0)]
amb = [k for k in used["key"].unique() if len(key2seeds[k]) > 1]
print(f"\nHMDD rows {len(hm)} | rows usable (descriptor match and a graph family) {len(used)} ({100 * len(used) / len(hm):.1f}%)")
print(f"HMDD miRNA keys used {used['key'].nunique()} | keys mapping to >1 seed family (arm ambiguity) {len(amb)} ({100 * len(amb) / used['key'].nunique():.1f}%)")

# 3. Disease -> HMDD record set. Two-descriptor diseases use the descriptor with fewer HMDD records (the more specific one).
rec = used.groupby("ui")["seeds"].apply(lambda s: set().union(*s)).to_dict()
cnt = used.groupby("ui").size().to_dict()
fm = pd.read_csv(EXT / "disease_mesh_map.csv").fillna("")
dsets = {}
for r in fm.itertuples():
    if not r.mesh_uis or r.flag == "non_disease":
        continue
    cand = [u for u in r.mesh_uis.split(";") if cnt.get(u, 0) > 0]
    if cand:
        dsets[r.disease_id] = rec[min(cand, key=lambda u: cnt[u])]
sz = np.array([len(v) for v in dsets.values()])
npairs = fm.set_index("disease_id")["n_pairs"]
print(f"\ndiseases with an HMDD family set {len(dsets)}/200 ({100 * npairs.reindex(list(dsets)).sum() / npairs.sum():.1f}% of pairs)")
print(f"disease family-set size: median {int(np.median(sz))}, 90th {int(np.percentile(sz, 90))}, max {sz.max()} | sets with <3 families: {int((sz < 3).sum())}")

# 4. Pre-registration of the details (appended only if the structure checks above held)
assert 380 <= len(universe) <= 450, "family universe is far from 416 - do not pre-register until this is understood"
EXT_YAML = PROJECT_ROOT / "configs" / "graph_ext.yaml"
cur = yaml.safe_load(EXT_YAML.read_text(encoding="utf-8"))
if "mirna_enrichment_details" not in cur:
    with open(EXT_YAML, "a", encoding="utf-8") as f:
        f.write('''
mirna_enrichment_details:   # pre-registered before any enrichment score or AUROC was computed
  hmdd: "v4.0 (2023.07) alldata_v4.txt, 53,553 records"
  disease_to_records: "records whose disease name equals the MeSH descriptor name of the disease (exact level, no descendants); for diseases with two descriptors, the descriptor with fewer HMDD records"
  mirna_mapping: "HMDD precursor name -> every graph family containing a mature miRNA with the same arm-less key (arm ambiguity accepted and counted)"
  universe: "the graph's miRNA families (families with >=1 working edge among the 1,582 circRNAs)"
  bound_set: "families with a working edge of the circRNA; no working edge -> score 0"
  score: "-log10 P(X >= k), hypergeometric survival with N=|universe|, n=|disease set|, K=|bound set|, k=|intersection|; score 0 when n=0, K=0 or k=0"
  controls: ["bound-set size only (expect 0.5)", "disease-set size only", "shuffled bound sets, 20 permutations"]
  pass_rule: "unchanged from diagnostics.pass_rule"
''')
    print("\nenrichment details pre-registered in", EXT_YAML.relative_to(PROJECT_ROOT))
else:
    print("\nenrichment details already pre-registered - left untouched")

print("\n===== OBSERVATIONS =====")
print(f"universe={len(universe)}; circRNAs with bound set={len(bound)}; HMDD rows usable={len(used)}/{len(hm)}; "
      f"ambiguous keys={len(amb)}/{used['key'].nunique()}; diseases with sets={len(dsets)}/200; "
      f"set size median={int(np.median(sz))}, 90th={int(np.percentile(sz, 90))}; sets<3={int((sz < 3).sum())}")

families with a working edge 416 (graph.yaml: 416) | unique circRNA-family edges 7540 (graph.yaml: 7540)
circRNAs with >=1 bound family 835/1582 | bound-set size median 5, max 107

HMDD rows 53553 | rows usable (descriptor match and a graph family) 42401 (79.2%)
HMDD miRNA keys used 469 | keys mapping to >1 seed family (arm ambiguity) 81 (17.3%)

diseases with an HMDD family set 191/200 (97.9% of pairs)
disease family-set size: median 58, 90th 179, max 305 | sets with <3 families: 4

enrichment details pre-registered in configs\graph_ext.yaml

===== OBSERVATIONS =====
universe=416; circRNAs with bound set=835; HMDD rows usable=42401/53553; ambiguous keys=81/469; diseases with sets=191/200; set size median=58, 90th=179; sets<3=4


**Interpretation:**
- The reconstruction matches the graph. The family universe is 416 and there are 7,540 unique circRNA-family edges, both identical to graph.yaml. The 835 circRNAs with a bound family match the earlier coverage table. So the mapping tables are built on exactly the graph that M1-M3 used.
- HMDD usability: 42,401 of 53,553 records (79.2%) have both a MeSH descriptor match and a graph family. Of 469 miRNA keys used, 81 (17.3%) map to more than one seed family, because HMDD names precursors and families are built from arms.
- Disease sets: 191 of 200 diseases have an HMDD family set (97.9% of pairs). The median set holds 58 families (14% of the universe), the 90th percentile 179 and the maximum 305 (73%). Only 4 sets have fewer than 3 families.
- The details are pre-registered in graph_ext.yaml, before any score exists.

**Decision:** compute the score and the controls exactly as pre-registered.

## Cell 11

**Objective:** compute the HMDD enrichment score for every pair (positives plus the frozen degree-matched negatives), evaluate it, run the three pre-registered controls, and apply the pass rule. Nothing is tuned.

In [27]:
# 09_disease_extension - Cell 11: HMDD enrichment diagnostic (details pre-registered in Cell 10; no parameters fitted)
from scipy.stats import hypergeom
from sklearn.metrics import roc_auc_score

EXT_YAML = PROJECT_ROOT / "configs" / "graph_ext.yaml"
cur = yaml.safe_load(EXT_YAML.read_text(encoding="utf-8"))
if "evaluation_as_run" not in cur:                     # documentation of procedure only, written before any score
    with open(EXT_YAML, "a", encoding="utf-8") as f:
        f.write('\nevaluation_as_run:   # procedure note; the pass rule is unchanged\n'
                '  host_gene_prior: "per-seed AUROC averaged over 3 seeds (as for M1-M3); unique-pair AUROC recomputed in Cell 11 of the notebook"\n'
                '  mirna_enrichment: "both versions reported; the pass rule is applied to the unique-pair AUROC, as written in diagnostics.evaluation"\n')

N_U = len(universe)
circ = pd.read_csv(D / "circrnas.csv"); all_c = circ["circbase_id"].tolist()
hostg = circ.set_index("circbase_id")["host_group"]
name2id = dict(zip(dis["disease"], dis["disease_id"]))
bset = lambda c, b=bound: b.get(c, set())
dset = lambda d: dsets.get(d, set())

def enrich(B, Dd):
    n, K = len(Dd), len(B)
    if n == 0 or K == 0:
        return 0.0
    k = len(B & Dd)
    if k == 0:
        return 0.0
    return float(-np.log10(max(hypergeom.sf(k - 1, N_U, n, K), 1e-300)))

# pairs: all positives (per seed) + the frozen degree-matched negatives (per seed)
neg = pd.read_csv(D / "negatives_cda.csv"); neg = neg[neg["scheme"] == "degree_matched"]
H = pd.concat([cda.assign(y=1, seed=s) for s in range(3)] + [neg[["seed", "circbase_id", "disease"]].assign(y=0)], ignore_index=True)
H["d"] = H["disease"].map(name2id)
Up = H[["circbase_id", "d"]].drop_duplicates()
sc = {(c, d): enrich(bset(c), dset(d)) for c, d in zip(Up["circbase_id"], Up["d"])}
key = list(zip(H["circbase_id"], H["d"]))
H["enrich"] = [sc[k] for k in key]                                     # PRE-REGISTERED score
H["bound_size"] = [len(bset(c)) for c in H["circbase_id"]]            # control 1 (circRNA only)
H["dis_size"] = [len(dset(d)) for d in H["d"]]                         # control 2 (disease only)
cov = np.array([len(bset(c)) > 0 and len(dset(d)) > 0 for c, d in key])
gid, ug = pd.factorize(H["circbase_id"].map(hostg))
uniq = ~H.duplicated(["circbase_id", "d", "y"]).values

def auc_seed(col, w=None, mask=None):
    out = []
    for s in range(3):
        m = H["seed"].values == s
        m = m if mask is None else (m & mask)
        y, x = H["y"].values[m], H[col].values[m]; ww = None if w is None else w[m]
        if len(np.unique(y)) < 2 or (ww is not None and (ww[y == 1].sum() == 0 or ww[y == 0].sum() == 0)):
            return np.nan
        out.append(roc_auc_score(y, x, sample_weight=ww))
    return float(np.mean(out))

def auc_uniq(col, w=None):
    y, x = H["y"].values[uniq], H[col].values[uniq]; ww = None if w is None else w[uniq]
    if len(np.unique(y)) < 2 or (ww is not None and (ww[y == 1].sum() == 0 or ww[y == 0].sum() == 0)):
        return np.nan
    return float(roc_auc_score(y, x, sample_weight=ww))

def with_ci(fn, n=1000, seed=0):
    rng, b = np.random.default_rng(seed), []
    for _ in range(n):
        w = np.bincount(rng.integers(0, len(ug), len(ug)), minlength=len(ug))[gid].astype(float)
        v = fn(w)
        if np.isfinite(v):
            b.append(v)
    lo, hi = np.percentile(b, [2.5, 97.5])
    return fn(None), lo, hi, len(b)

rows = []
for label, fn in (("HMDD enrichment, unique pairs (PRE-REGISTERED)", lambda w=None: auc_uniq("enrich", w)),
                  ("HMDD enrichment, per-seed mean (as M1-M3)", lambda w=None: auc_seed("enrich", w)),
                  ("  covered pairs only (per-seed)", lambda w=None: auc_seed("enrich", w, cov)),
                  ("control: bound-set size only (expect 0.5)", lambda w=None: auc_seed("bound_size", w)),
                  ("control: disease-set size only", lambda w=None: auc_seed("dis_size", w))):
    a, lo, hi, nb = with_ci(fn)
    rows.append({"score": label, "AUROC": round(a, 3), "ci95": f"[{lo:.3f}, {hi:.3f}]", "boot_ok": nb})
print(pd.DataFrame(rows).to_string(index=False))

# control 3: shuffled bound sets (keeps every circRNA's number of bound families, breaks which families)
rng, ctrl = np.random.default_rng(12345), []
for _ in range(20):
    perm = dict(zip(all_c, rng.permutation(all_c)))
    cache = {k: enrich(bset(perm[k[0]]), dset(k[1])) for k in sc}
    H["shuf"] = [cache[k] for k in key]
    ctrl.append(auc_seed("shuf"))
print(f"\nshuffled bound-set control: mean AUROC {np.mean(ctrl):.3f} (min {np.min(ctrl):.3f}, max {np.max(ctrl):.3f}) over 20 shuffles")
print(f"pairs with both a bound set and a disease set: {100 * cov.mean():.1f}% | non-zero score: {100 * (H['enrich'] > 0).mean():.1f}%")

# host-gene prior under the unique-pair definition (only if Cell 8's results are still in memory)
try:
    Eu = E.drop_duplicates(["circbase_id", "d", "y"])
    print(f"host-gene prior, unique-pair AUROC: {roc_auc_score(Eu['y'], Eu['prior']):.3f}")
except NameError:
    print("(Cell 8 state not in memory: host-gene unique-pair AUROC not recomputed)")

# pre-registered rule: AUROC >= 0.55 and lower 95% bound > 0.527 (unique pairs)
a = rows[0]["AUROC"]; lo = float(rows[0]["ci95"].strip("[]").split(",")[0])
ok = a >= 0.55 and lo > 0.527
print("\nPRE-REGISTERED RULE (AUROC >= 0.55 and lower bound > 0.527): " + ("PASS - the relation may be added" if ok else "NOT met - do not add the relation"))

print("\n===== OBSERVATIONS =====")
for r in rows:
    print(f"{r['score'].strip()}: {r['AUROC']} {r['ci95']}")
print(f"shuffled control {np.mean(ctrl):.3f}; non-zero scores {100 * (H['enrich'] > 0).mean():.1f}%; rule passed: {ok}")

                                         score  AUROC           ci95  boot_ok
HMDD enrichment, unique pairs (PRE-REGISTERED)  0.509 [0.502, 0.517]     1000
     HMDD enrichment, per-seed mean (as M1-M3)  0.507 [0.500, 0.513]     1000
                 covered pairs only (per-seed)  0.509 [0.495, 0.523]     1000
     control: bound-set size only (expect 0.5)  0.500 [0.500, 0.500]     1000
                control: disease-set size only  0.522 [0.505, 0.540]     1000

shuffled bound-set control: mean AUROC 0.504 (min 0.499, max 0.511) over 20 shuffles
pairs with both a bound set and a disease set: 53.2% | non-zero score: 41.1%
host-gene prior, unique-pair AUROC: 0.507

PRE-REGISTERED RULE (AUROC >= 0.55 and lower bound > 0.527): NOT met - do not add the relation

===== OBSERVATIONS =====
HMDD enrichment, unique pairs (PRE-REGISTERED): 0.509 [0.502, 0.517]
HMDD enrichment, per-seed mean (as M1-M3): 0.507 [0.500, 0.513]
covered pairs only (per-seed): 0.509 [0.495, 0.523]
control: bound-set s

**Interpretation:**
- Primary (unique pairs): AUROC 0.509, 95% CI [0.502, 0.517]. The pre-registered rule (AUROC ≥ 0.55 and a lower bound above 0.527) fails on both criteria. The per-seed version gives 0.507 [0.500, 0.513]. The two definitions agree, and the unique-pair AUROC of the host-gene prior is also 0.507, so the deviation I flagged did not change anything.
- The real score is not distinguishable from shuffled sets. Shuffling which families each circRNA binds, while keeping how many it binds, gives a mean of 0.504 (range 0.499-0.511). The specific miRNA overlap carries no measurable information about disease association here.
- The bound-set-size control is exactly 0.500, which confirms again that the frozen negatives are matched per circRNA.
- The disease-set-size control scores higher than the enrichment itself: 0.522, CI [0.505, 0.540]. A disease-only score, with no circRNA information, separates positives from degree-matched negatives slightly better than chance, and better than the pre-registered score.
- Coverage: 53.2% of pairs have both a bound set and a disease set, and 41.1% have a non-zero score.

**Decision:** the stop rule applies. The miRNA-disease relation is not added. This closes all three tests: MeSH features (+0.003, CI includes 0), host-gene prior (0.507) and HMDD enrichment (0.509).

In [3]:
# 09_disease_extension - Cell 12: final record of the extension in configs/graph_ext.yaml (appended, never edits earlier text)
EXT_YAML = PROJECT_ROOT / "configs" / "graph_ext.yaml"
cur = yaml.safe_load(EXT_YAML.read_text(encoding="utf-8"))
if "diagnostic_outcomes_final" not in cur:
    with open(EXT_YAML, "a", encoding="utf-8") as f:
        f.write('''
diagnostic_outcomes_final:   # supersedes any earlier outcome note; the pre-registrations above are unchanged
  host_gene_prior: "AUROC 0.507 (95% CI 0.500-0.515); shuffled-gene control 0.502; rule not met; relation not added"
  mirna_enrichment: "HMDD v4.0; unique-pair AUROC 0.509 (95% CI 0.502-0.517); shuffled bound sets 0.504; disease-set-size control 0.522 (0.505-0.540); rule not met; relation not added"
  mesh_features: "M3_mesh - M3 = +0.003 (95% CI -0.007 to 0.012), degree-matched; rule not met; main results stand"
  not_run: ["is_a edges", "learned residual", "strict-collision sensitivity (bounded at about 0.003 by the Cell 4 argument)"]
  extension_conclusion: "No tested disease-side input exceeded 0.534 on degree-matched negatives; the extension is closed"
''')
print(yaml.safe_load(EXT_YAML.read_text(encoding="utf-8"))["diagnostic_outcomes_final"]["extension_conclusion"])

No tested disease-side input exceeded 0.534 on degree-matched negatives; the extension is closed


## Cell 13 

In [17]:
# 09_disease_extension - Cell 13: circBase ID -> circAtlas ID -> tissue-detection profile (replaces the liftover version)
import numpy as np, pandas as pd

CA = PROJECT_ROOT / "data" / "raw" / "circatlas"
MAPFILE = CA / "circatlas_id_mapping_to_circatlass.csv"      # <- your conversion output (needs columns input_id, status, circAtlas_ID)
TISFILE = CA / "circatlas_unique_circRNAs.csv"
OUTX = D / "extension_expr"; OUTX.mkdir(exist_ok=True)

nodes_c = pd.read_csv(D / "graph" / "nodes_circrna.csv", keep_default_na=False)
ids = nodes_c["circbase_id"].tolist()

# 1. conversion table (utf-8-sig drops a byte-order mark; delimiter read from the header line)
head = open(MAPFILE, encoding="utf-8-sig").readline()
sep = "\t" if head.count("\t") >= head.count(",") else ","
mp = pd.read_csv(MAPFILE, sep=sep, encoding="utf-8-sig", dtype=str, keep_default_na=False)
mp.columns = [c.strip() for c in mp.columns]
need = {"input_id", "status", "circAtlas_ID"} - set(mp.columns)
assert not need, f"missing columns {need}; found {list(mp.columns)}"
mp = (mp.sort_values("status", key=lambda s: s.ne("matched"), kind="stable")     # a 'matched' row beats an older 'error' row
        .drop_duplicates("input_id").set_index("input_id"))
status = mp["status"].reindex(ids).fillna("not_queried")
ca_id = mp["circAtlas_ID"].reindex(ids).fillna("")
ca_id = ca_id.where(~ca_id.isin(["-", ""]), "")
print("conversion status:", status.value_counts().to_dict())
print(f"circBase IDs with a circAtlas ID: {int((ca_id != '').sum())} / {len(ids)}")
rerun = [i for i, s in zip(ids, status) if s not in ("matched", "no_entry")]
pd.Series(rerun, name="input_id").to_csv(OUTX / "ids_to_rerun.csv", index=False)
print(f"IDs still to query (errors or not queried): {len(rerun)} -> {OUTX / 'ids_to_rerun.csv'}")

# 2. tissue file (one row per circAtlas circRNA; the 'tissues' column lists ALL tissues of that circRNA)
ca_all = pd.read_csv(TISFILE, dtype=str, keep_default_na=False, encoding="utf-8-sig", usecols=["species", "name", "tissues", "ntis", "nsam"])
ca_all = ca_all[ca_all["species"].str.lower() == "human"]
n_conf = int((ca_all.groupby("name")["tissues"].nunique() > 1).sum())
print(f"tissue file: {len(ca_all):,} human rows | {ca_all['name'].nunique():,} distinct names | names with differing tissue lists between exports: {n_conf}")
tab = ca_all.drop_duplicates("name").set_index("name")
tab["tl"] = tab["tissues"].str.split(";").apply(lambda L: sorted({t.strip() for t in L if t.strip()}))
n_bad = int((tab["tl"].str.len() != pd.to_numeric(tab["ntis"], errors="coerce")).sum())
TISSUES = sorted({t for L in tab["tl"] for t in L})
print(f"integrity: rows where len(tissue list) != ntis: {n_bad:,} | tissues ({len(TISSUES)}): {TISSUES}")

# 3. profiles
tl, ns = tab["tl"].to_dict(), tab["nsam"].to_dict()
col = {t: j for j, t in enumerate(TISSUES)}
P = np.zeros((len(ids), len(TISSUES)), np.float32); nsam = np.zeros(len(ids), np.float32)
missing = np.ones(len(ids), bool)
for k, cid in enumerate(ca_id):
    if cid and cid in tl:
        missing[k] = False
        for t in tl[cid]:
            P[k, col[t]] = 1.0
        nsam[k] = pd.to_numeric(ns[cid], errors="coerce") if ns[cid] not in ("", "unknown") else 0
np.save(OUTX / "tissue_profile.npy", P); np.save(OUTX / "tissue_missing.npy", missing); np.save(OUTX / "tissue_nsam.npy", nsam)
pd.DataFrame({"circbase_id": ids, "circatlas_id": ca_id.to_numpy(), "has_profile": ~missing}).to_csv(OUTX / "circatlas_map.csv", index=False)

cov = 1 - missing.mean()
nt = P[~missing].sum(1)
print(f"\nwith a tissue profile: {int((~missing).sum())} / {len(ids)} ({100 * cov:.1f}%)")
print(f"  have a circAtlas ID but are absent from the tissue file: {int(((ca_id != '') & missing).sum())}")
print(f"  tissues per profiled circRNA: median {np.median(nt):.0f} | exactly 1 tissue {100 * (nt == 1).mean():.0f}% | >= 3 tissues {100 * (nt >= 3).mean():.0f}%")
print("  circRNAs per tissue:", {t: int(v) for t, v in zip(TISSUES, P.sum(0))})
print("COVERAGE GATE (>= 60%):", "PASSED" if cov >= 0.60 else "NOT YET MET" + (" - rerun the failed IDs first" if rerun else " - stop and report"))

conversion status: {'matched': 1436, 'no_entry': 146}
circBase IDs with a circAtlas ID: 1055 / 1582
IDs still to query (errors or not queried): 0 -> d:\Queens_university\Project\Working\Notebook\circmtl\data\processed\dataset_v1\extension_expr\ids_to_rerun.csv
tissue file: 768,986 human rows | 768,986 distinct names | names with differing tissue lists between exports: 0
integrity: rows where len(tissue list) != ntis: 0 | tissues (33): ['Adipose', 'Adrenal gland', 'Bladder', 'Blood', 'Bone Marrow', 'Brain', 'Breast', 'Cervix', 'Colon', 'Esophagus', 'Heart', 'Kidney', 'Liver', 'Lung', 'Lymph node', 'Mammary gland', 'Ovary', 'Pancreas', 'Placenta', 'Prostate', 'Retina', 'Skeletal Muscle', 'Skin', 'Small Intestine', 'Spinal Cord', 'Spleen', 'Stomach', 'Testis', 'Thymus', 'Thyroid', 'Tonsil', 'Trachea', 'Uterus']

with a tissue profile: 1055 / 1582 (66.7%)
  have a circAtlas ID but are absent from the tissue file: 0
  tissues per profiled circRNA: median 26 | exactly 1 tissue 4% | >= 3 tiss

In [18]:
# 09_disease_extension - Cell 14: tissue-profile diagnostic (no model; every statistic from TRAINING circRNAs only)
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
try:
    gd
except NameError:
    import sys; sys.path.insert(0, str(PROJECT_ROOT / "src"))
    from circmtl.graph import GraphData
    gd = GraphData(D, D / "graph")

OUTX = D / "extension_expr"
P0 = np.load(OUTX / "tissue_profile.npy").astype(np.float64)
has = ~np.load(OUTX / "tissue_missing.npy")
nC, T = P0.shape; nD = gd.N["disease"]
N_BOOT, SEEDS, FOLDS = 1000, range(3), range(5)

def cos_rows(A, B):                                   # row-wise cosine; a zero vector gives 0
    na, nb = np.linalg.norm(A, axis=1), np.linalg.norm(B, axis=1)
    return np.where((na > 0) & (nb > 0), (A * B).sum(1) / np.maximum(na * nb, 1e-12), 0.0)

def fold_scores(P, fd, pairs):
    tr = fd["train"][has[fd["train"]]]
    mu = P[tr].mean(0)                                # mean profile of the fold's training circRNAs
    dt = fd["disease_train"]; dt = dt[has[dt[:, 0]]]
    S = np.zeros((nD, T)); cnt = np.zeros(nD)
    np.add.at(S, dt[:, 1], P[dt[:, 0]]); np.add.at(cnt, dt[:, 1], 1)
    Cen = np.where(cnt[:, None] > 0, S / np.maximum(cnt[:, None], 1), 0.0)     # disease = mean training profile
    c, d = pairs[:, 0], pairs[:, 1]
    ok = has[c] & (cnt[d] > 0)
    Pc, Cd = P[c], Cen[d]
    plain = np.where(ok, cos_rows(Pc, Cd), 0.0)                                # sensitivity
    cent = np.where(ok, cos_rows(Pc - mu, Cd - mu), 0.0)                       # PRIMARY: centred cosine
    dis = np.where(cnt[d] > 0, Cd.sum(1), 0.0)                                 # control: disease side only
    return plain, cent, dis

rng = np.random.default_rng(2026)
idx = np.flatnonzero(has); Pshuf = P0.copy(); Pshuf[idx] = P0[rng.permutation(idx)]   # control: profiles shuffled across circRNAs

rows = []
for s in SEEDS:
    for k in FOLDS:
        fd = gd.load_fold(s, k)
        pos, neg = fd["disease_test"], fd["neg_test_degree_matched"]
        pairs = np.vstack([pos, neg]); y = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
        plain, cent, dis = fold_scores(P0, fd, pairs)
        _, shuf, _ = fold_scores(Pshuf, fd, pairs)
        df = pd.DataFrame({"seed": s, "fold": k, "c": pairs[:, 0], "d": pairs[:, 1], "y": y,
                           "plain": plain, "centred": cent, "shuffled": shuf, "disease_only": dis})
        for col in ("plain", "centred", "shuffled", "disease_only"):
            df[col] = df[col].rank(pct=True)                                   # rank-normalise within seed-fold, as for the models
        rows.append(df)
R = pd.concat(rows, ignore_index=True)
R["has"] = has[R["c"].to_numpy()]
yv, hv = R["y"].to_numpy(), R["has"].to_numpy()
seed_ix = [np.flatnonzero(R["seed"].to_numpy() == s) for s in SEEDS]

def auc_w(col, w=None, mask=None):
    v, out = R[col].to_numpy(), []
    for ix in seed_ix:
        if mask is not None: ix = ix[mask[ix]]
        out.append(roc_auc_score(yv[ix], v[ix], sample_weight=None if w is None else w[ix]))
    return out

summ = []
for col in ("centred", "plain", "shuffled", "disease_only"):
    a_all, a_has = auc_w(col), auc_w(col, mask=hv)
    summ.append({"score": col, "AUROC_all_pairs": f"{np.mean(a_all):.3f} ({min(a_all):.3f}-{max(a_all):.3f})",
                 "AUROC_profiled_only": f"{np.mean(a_has):.3f}"})
print(f"test pairs: {len(R) // len(list(SEEDS)):,} per seed | with a profile: {100 * hv.mean():.1f}%")
print(pd.DataFrame(summ).to_string(index=False))

ug, gi = np.unique(gd.groups[R["c"].to_numpy()], return_inverse=True)
rb = np.random.default_rng(0)
B = {"centred": [], "shuffled": []}
for _ in range(N_BOOT):
    w = np.bincount(rb.integers(0, len(ug), len(ug)), minlength=len(ug))[gi].astype(float)
    for col in B: B[col].append(np.mean(auc_w(col, w=w)))
lo = {c: np.percentile(v, 2.5) for c, v in B.items()}; hi = {c: np.percentile(v, 97.5) for c, v in B.items()}
for c in B: print(f"{c:9s} AUROC {np.mean(auc_w(c)):.3f}  95% CI over host groups [{lo[c]:.3f}, {hi[c]:.3f}]")
a = np.mean(auc_w("centred"))
print("\nPASS RULE (centred, all test pairs): AUROC >= 0.55 and lower bound > 0.527 ->",
      "PASSED" if (a >= 0.55 and lo["centred"] > 0.527) else "NOT MET - report and close")
R.to_csv(OUTX / "tissue_diagnostic_pairs.csv", index=False)

test pairs: 4,274 per seed | with a profile: 71.6%
       score     AUROC_all_pairs AUROC_profiled_only
     centred 0.542 (0.537-0.548)               0.552
       plain 0.510 (0.509-0.512)               0.524
    shuffled 0.491 (0.485-0.498)               0.491
disease_only 0.490 (0.486-0.496)               0.516
centred   AUROC 0.542  95% CI over host groups [0.531, 0.553]
shuffled  AUROC 0.491  95% CI over host groups [0.478, 0.504]

PASS RULE (centred, all test pairs): AUROC >= 0.55 and lower bound > 0.527 -> NOT MET - report and close
